In [6]:
import pandas as pd

In [7]:
from IPython.display import display
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', True)

In [8]:
NER = pd.read_excel("NER.xlsx")

In [9]:
NER.shape

(129249, 9)

In [10]:
NER.head()

,RIC,sdate,edate,Headlines,Cleaned_Headlines,Tokens,NER,NER2,NER_combined
0,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,"CIF/FOB Gulf Grain-Corn barge basis steady to lower, soybeans mixed",CIF FOB Gulf Grain Corn barge basis steady to lower soybeans mixed,"['cif', 'fob', 'gulf', 'grain', 'corn', 'barge', 'basis', 'steady', 'to', 'lower', 'soybeans', 'mixed']","[('CIF', 'ORG')]","[('CIF FOB', 'ORG')]","[('CIF', 'ORG'), ('CIF FOB', 'ORG')]"
1,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,"Export Summary-Philippine importer buys feed wheat, South Korea seeks soybeans",Export Summary Philippine importer buys feed wheat South Korea seeks soybeans,"['export', 'summary', 'philippine', 'importer', 'buys', 'feed', 'wheat', 'south', 'korea', 'seeks', 'soybeans']","[('Philippine', 'NORP'), ('South Korea', 'GPE')]","[('Philippine', 'NORP'), ('South Korea', 'GPE')]","[('Philippine', 'NORP'), ('South Korea', 'GPE')]"
2,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,DJ Thailand Corn Weather - Nov 9,Thailand Corn Weather,"['thailand', 'corn', 'weather']","[('Thailand', 'GPE')]","[('Thailand', 'GPE')]","[('Thailand', 'GPE')]"
3,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,DJ Northeast China Corn Weather - Nov 9,Northeast China Corn Weather,"['northeast', 'china', 'corn', 'weather']","[('Northeast', 'ORG'), ('China', 'GPE')]","[('China', 'GPE')]","[('Northeast', 'ORG'), ('China', 'GPE')]"
4,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,Corn: Cargo market without buyers,Corn Cargo market without buyers,"['corn', 'cargo', 'market', 'without', 'buyers']",[],[],[]


In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

# Load FinBERT model
tokenizer = AutoTokenizer.from_pretrained('yiyanghkust/finbert-tone')
model = AutoModelForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone')

In [2]:
def get_finbert_sentiment(text):
    if not isinstance(text, str) or text.strip() == '':
        return {'positive': np.nan, 'neutral': np.nan, 'negative': np.nan}
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1).detach().cpu().numpy()[0]
    return {
        'positive': probs[0],
        'neutral': probs[1],
        'negative': probs[2]
    }


In [11]:
# Apply sentiment analysis
sentiments = NER['Cleaned_Headlines'].apply(get_finbert_sentiment)

# Expand into separate columns
sentiment_df = pd.DataFrame(sentiments.tolist())

# Merge sentiment columns back to your main NER dataframe
NER = pd.concat([NER, sentiment_df], axis=1)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [12]:
NER.head()

,RIC,sdate,edate,Headlines,Cleaned_Headlines,Tokens,NER,NER2,NER_combined,positive,neutral,negative
0,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,"CIF/FOB Gulf Grain-Corn barge basis steady to lower, soybeans mixed",CIF FOB Gulf Grain Corn barge basis steady to lower soybeans mixed,"['cif', 'fob', 'gulf', 'grain', 'corn', 'barge', 'basis', 'steady', 'to', 'lower', 'soybeans', 'mixed']","[('CIF', 'ORG')]","[('CIF FOB', 'ORG')]","[('CIF', 'ORG'), ('CIF FOB', 'ORG')]",0.074360,9.236270e-01,0.002013
1,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,"Export Summary-Philippine importer buys feed wheat, South Korea seeks soybeans",Export Summary Philippine importer buys feed wheat South Korea seeks soybeans,"['export', 'summary', 'philippine', 'importer', 'buys', 'feed', 'wheat', 'south', 'korea', 'seeks', 'soybeans']","[('Philippine', 'NORP'), ('South Korea', 'GPE')]","[('Philippine', 'NORP'), ('South Korea', 'GPE')]","[('Philippine', 'NORP'), ('South Korea', 'GPE')]",0.999994,6.149971e-07,0.000005
2,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,DJ Thailand Corn Weather - Nov 9,Thailand Corn Weather,"['thailand', 'corn', 'weather']","[('Thailand', 'GPE')]","[('Thailand', 'GPE')]","[('Thailand', 'GPE')]",0.390487,2.177151e-04,0.609296
3,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,DJ Northeast China Corn Weather - Nov 9,Northeast China Corn Weather,"['northeast', 'china', 'corn', 'weather']","[('Northeast', 'ORG'), ('China', 'GPE')]","[('China', 'GPE')]","[('Northeast', 'ORG'), ('China', 'GPE')]",0.722697,5.726411e-04,0.276730
4,COR,2021-10-16 00:00:00,2021-11-10 00:00:00,Corn: Cargo market without buyers,Corn Cargo market without buyers,"['corn', 'cargo', 'market', 'without', 'buyers']",[],[],[],0.999573,7.261986e-06,0.000420


In [14]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from joblib import Parallel, delayed
import multiprocessing

# Step 1: Label function
def label_sentiment_single(row):
    if row['positive'] >= row['neutral'] and row['positive'] >= row['negative']:
        return 1
    elif row['negative'] >= row['positive'] and row['negative'] >= row['neutral']:
        return -1
    else:
        return 0

# Step 2: Parallelize over positive, neutral, negative
num_cores = multiprocessing.cpu_count()

labels = Parallel(n_jobs=num_cores)(
    delayed(label_sentiment_single)(row) for _, row in NER[['positive', 'neutral', 'negative']].iterrows()
)

NER['Sentiment_Label'] = labels

# Step 3: Aggregate
ric_sentiment = NER.groupby('RIC')['Sentiment_Label'].mean().reset_index()
ric_sentiment.rename(columns={'Sentiment_Label': 'Avg_Sentiment_Label'}, inplace=True)


In [15]:
ric_sentiment

,RIC,Avg_Sentiment_Label
0,000150.KS,0.793103
1,002230.SZ,0.666667
2,002281.SZ,0.454545
3,002405.SZ,0.428571
4,002432.SZ,1.000000
...,...,...
2211,ZM.OQ,0.545000
2212,ZS.OQ,0.727273
2213,ZTS.N,-0.375000
2214,ZUO.N,0.500000


In [17]:
ric_sentiment.to_excel("Sentiment.xlsx", index=False)